## Mart_Table (mart_review_sentiment) GOLD LAYER INSERTION

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

### Importing Libraries

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, avg, count, countDistinct,
    round as spark_round, col, when, lag, try_divide
)
from pyspark.sql import Window

#### Load and Join Data

In [0]:

# Load tables
df_fact = spark.table("olist_ecommerce_project.gold.fact_orders")
df_date = spark.table("olist_ecommerce_project.gold.dim_date")
df_order_items = spark.table("olist_ecommerce_project.silver.slv_order_items")
df_products = spark.table("olist_ecommerce_project.gold.dim_products")
df_sellers = spark.table("olist_ecommerce_project.gold.dim_sellers")

# Join: fact → date → order_items (to get product_id and seller_id) → products → sellers
df_sentiment = (
    df_fact
    .join(df_date, on="date_key", how="inner")
    .join(df_order_items, on="order_id", how="inner")
    .join(df_products, on="product_id", how="left")
    .join(df_sellers, on="seller_id", how="left")
    .filter(col("review_score").isNotNull())  # Only orders with reviews
)

print("Sentiment data rows:", df_sentiment.count())
df_sentiment.select("order_id", "review_score", "product_category_name_english", "seller_state").show(3, truncate=False)

#### Aggregate by Month

In [0]:
# Group by month and calculate review metrics
df_sentiment_agg = (
    df_sentiment
    .groupBy("year", "month_number", "month_name")
    .agg(
        count("order_id").alias("total_reviews"),
        spark_round(avg("review_score"), 2).alias("avg_review_score"),
        spark_sum(when(col("review_score") >= 4, 1).otherwise(0)).alias("positive_reviews"),
        spark_sum(when(col("review_score") == 3, 1).otherwise(0)).alias("neutral_reviews"),
        spark_sum(when(col("review_score") <= 2, 1).otherwise(0)).alias("negative_reviews"),
        spark_sum(when(col("has_comment") == True, 1).otherwise(0)).alias("reviews_with_comments")
    )
    .orderBy("year", "month_number")
)

print("Sentiment aggregated rows:", df_sentiment_agg.count())
df_sentiment_agg.show(5, truncate=False)

#### Calculate Sentiment Percentages and Weighted Score

In [0]:
# Calculate sentiment distribution percentages
df_sentiment_dist = (
    df_sentiment_agg
    .withColumn(
        "positive_review_pct",
        spark_round(col("positive_reviews") / col("total_reviews") * 100, 2)
    )
    .withColumn(
        "neutral_review_pct",
        spark_round(col("neutral_reviews") / col("total_reviews") * 100, 2)
    )
    .withColumn(
        "negative_review_pct",
        spark_round(col("negative_reviews") / col("total_reviews") * 100, 2)
    )
    .withColumn(
        "reviews_with_comments_pct",
        spark_round(col("reviews_with_comments") / col("total_reviews") * 100, 2)
    )
)

# Calculate weighted sentiment score
# Positive = +1, Neutral = 0, Negative = -1
df_sentiment_dist = (
    df_sentiment_dist
    .withColumn(
        "weighted_sentiment_score",
        spark_round(
            (col("positive_reviews") - col("negative_reviews")) / col("total_reviews"),
            3
        )
    )
)

print("Sentiment distribution calculated")
df_sentiment_dist.select(
    "month_name",
    "avg_review_score",
    "positive_review_pct",
    "negative_review_pct",
    "weighted_sentiment_score"
).show(5, truncate=False)

#### Add Sentiment vs Delivery Correlation

In [0]:
# Measure correlation: do late deliveries get lower scores?
df_sentiment_delivery = (
    df_sentiment
    .groupBy("year", "month_number", "month_name")
    .agg(
        spark_round(
            avg(when(col("is_late") == True, col("review_score")).otherwise(None)),
            2
        ).alias("avg_score_late_orders"),
        spark_round(
            avg(when(col("is_late") == False, col("review_score")).otherwise(None)),
            2
        ).alias("avg_score_ontime_orders"),
        spark_round(
            try_divide(
                spark_sum(when((col("is_late") == True) & (col("review_score") <= 2), 1).otherwise(0)),
                spark_sum(when(col("is_late") == True, 1).otherwise(0))
            ) * 100,
            2
        ).alias("late_orders_negative_review_pct")
    )
    .orderBy("year", "month_number")
)

print("Delivery correlation calculated")
df_sentiment_delivery.show(5, truncate=False)

#### Calculate Sentiment Trend (MoM)

In [0]:
# Calculate month-over-month sentiment change
window_trend = Window.orderBy("year", "month_number")

df_sentiment_trend = (
    df_sentiment_dist
    .withColumn(
        "prev_month_avg_score",
        lag("avg_review_score").over(window_trend)
    )
    .withColumn(
        "sentiment_trend_change",
        spark_round(col("avg_review_score") - col("prev_month_avg_score"), 3)
    )
)

print("Sentiment trend calculated")
df_sentiment_trend.select(
    "month_name",
    "avg_review_score",
    "prev_month_avg_score",
    "sentiment_trend_change"
).show(10, truncate=False)

#### Combine and Write Final Mart

In [0]:
# Combine all aggregations
df_mart_sentiment = (
    df_sentiment_dist
    .join(
        df_sentiment_delivery,
        on=["year", "month_number", "month_name"],
        how="left"
    )
    .join(
        df_sentiment_trend.select("year", "month_number", "sentiment_trend_change"),
        on=["year", "month_number"],
        how="left"
    )
)

# Select final columns
df_mart_sentiment = (
    df_mart_sentiment.select(
        "year",
        "month_number",
        "month_name",
        "total_reviews",
        "avg_review_score",
        "positive_reviews",
        "neutral_reviews",
        "negative_reviews",
        "positive_review_pct",
        "neutral_review_pct",
        "negative_review_pct",
        "reviews_with_comments_pct",
        "weighted_sentiment_score",
        "avg_score_late_orders",
        "avg_score_ontime_orders",
        "late_orders_negative_review_pct",
        "sentiment_trend_change"
    )
    .orderBy("year", "month_number")
)

print("mart_review_sentiment rows:", df_mart_sentiment.count())
df_mart_sentiment.select(
    "month_name",
    "avg_review_score",
    "positive_review_pct",
    "weighted_sentiment_score",
    "avg_score_late_orders",
    "avg_score_ontime_orders",
    "sentiment_trend_change"
).show(10, truncate=False)

# Write to Gold
(
    df_mart_sentiment.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.mart_review_sentiment")
)

print("mart_review_sentiment written successfully")

##### 24 rows, all metrics present:

- Sentiment scores and distribution ✅
- Late vs on-time delivery correlation ✅
- Month-over-month sentiment trends ✅